## Notebook to demo HotSpot Analysis.

Data was download from [here](https://www.arcgis.com/home/item.html?id=71a65d35688a4502b123cbdfc99afdee). Note that the spatial reference of the data is in "ESRI:102726".

References:

- https://pro.arcgis.com/en/pro-app/latest/tool-reference/spatial-statistics/what-is-a-z-score-what-is-a-p-value.htm
- https://pro.arcgis.com/en/pro-app/latest/tool-reference/spatial-statistics/h-how-hot-spot-analysis-getis-ord-gi-spatial-stati.htm

In [1]:
from itertools import product

import geofunctions as S
import geopandas as gp
import pyspark.sql.functions as F
import xyzservices.providers as xyz

In [2]:
pts = (
    spark.read.format("shp")
    .options(path="RegressionTutorial/RegressionExercise/Data/911Calls.shp")
    .load()
)

### Get count per cell.

In [3]:
cell = 1000

qr_ln = (
    pts.select(S.st_xy("shape").alias("xy"))
    .select(
        S.st_xtoq("xy.x", cell),
        S.st_ytor("xy.y", cell),
    )
    .groupBy("q", "r")
    .count()
    .withColumnRenamed("count", "c")
)

### Define a neighborhood of a cell.

In [4]:
data = list(product(range(-3, 4), range(-3, 4)))
prod = spark.createDataFrame(data, "i int,j int")

### Broadcast the cell to its neighboors.

In [5]:
qr_nn = (
    qr_ln
    .crossJoin(F.broadcast(prod))
    .select(F.expr("q+i as q"), F.expr("r+j as r"), "c")
    .groupBy("q", "r")
    .agg(F.sum("c").alias("n"))
)

### Get the overall statistics.

In [6]:
rows = qr_nn.select("n").summary("mean", "stddev").select("n").collect()
mu = F.lit(float(rows[0].n))
sd = F.lit(float(rows[1].n))

23/08/02 12:47:13 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [8]:
z = (
    qr_nn.join(qr_ln, ["q", "r"])
    .select(
        S.st_qtox("q", cell),
        S.st_rtoy("r", cell),
        "c",
        "n",
        ((F.col("n") - mu) / sd).alias("z"),
    )
    .select(
        S.st_cell("x", "y", cell).alias("geometry"),
        "c",
        "n",
        F.greatest(F.lit(-3.0), F.least(F.lit(3.0), "z")).alias("z"),
    )
    .orderBy("z")
)

In [9]:
pdf = z.toPandas()

In [10]:
pdf.geometry = gp.GeoSeries.from_wkb(pdf.geometry)
gdf = gp.GeoDataFrame(pdf, crs="ESRI:102726")
gdf.sindex;

In [11]:
gdf.explore("z", cmap="coolwarm", vmin=-3.0, vmax=3.0, tiles=xyz.Esri.WorldGrayCanvas)